# 09 — Multi-Seed Robustness of Transformation-Based Synthetic Augmentation

## Objective

This notebook evaluates whether the performance improvement produced by transformation-based synthetic augmentation is reproducible across independent model-training seeds.

Two training configurations are compared:

1. **10% real-only:** 1,150 balanced real observations.
2. **10% real + synthetic 1:1:** 1,150 real and 1,150 synthetic observations.

The official training samples, synthetic dataset, validation partition, and test partition remain fixed. Only model initialization, training-set shuffling, and dropout randomness change across seeds.

The experiment uses five seeds:

- 42
- 52
- 62
- 72
- 82

Seed 42 reuses the completed results from Notebooks 06 and 08. The remaining seeds are trained in this notebook.

Each model selects its classification threshold using only the validation partition. The test set is evaluated only after the corresponding threshold has been locked.

The objective is to determine whether augmentation produces a consistent paired improvement in balanced accuracy, macro-F1, minority-class performance, and ROC-AUC.

## 1. Safe Configuration and Run Inventory

The notebook defaults to `RUN_NEW_TRAINING = False`. It validates all source files and inventories every seed–configuration pair before any training can occur. Existing checkpoints and evaluation artifacts are detected and never overwritten. After a kernel restart, running all cells reloads the completed artifacts and reproduces the summaries without fitting new models.


In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from scipy import stats

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from tensorflow.keras import (
    layers,
    models,
    regularizers
)

In [ ]:
BATCH_SIZE = 64
MAX_EPOCHS = 50

REAL_SUBSET = "10_percent"
SYNTHETIC_RATIO = 1.0

EXPERIMENT_SEEDS = [
    42,
    52,
    62,
    72,
    82
]

REFERENCE_SEED = 42

NEW_TRAINING_SEEDS = [
    seed
    for seed in EXPERIMENT_SEEDS
    if seed != REFERENCE_SEED
]

CONFIGURATIONS = {
    "real_only": {
        "display_name":
            "10% real-only",

        "use_synthetic":
            False
    },

    "real_plus_synthetic": {
        "display_name":
            "10% real + synthetic 1:1",

        "use_synthetic":
            True
    }
}

# Safe default for notebook inspection.
# We will change this only after all checks pass.
RUN_NEW_TRAINING = False

print(
    "TensorFlow version:",
    tf.__version__
)

print(
    "Available GPUs:",
    tf.config.list_physical_devices(
        "GPU"
    )
)

print(
    "Experiment seeds:",
    EXPERIMENT_SEEDS
)

print(
    "Reference seed:",
    REFERENCE_SEED
)

print(
    "New seeds:",
    NEW_TRAINING_SEEDS
)

print(
    "Run new training:",
    RUN_NEW_TRAINING
)

In [ ]:
OFFICIAL_DATA_DIR = Path(
    "../data/processed/official_split"
)

LIMITED_DATA_DIR = Path(
    "../data/processed/limited_subsets"
)

SYNTHETIC_DATASET_DIR = (
    Path("../data/processed/synthetic_subsets")
    / (
        "10_percent_signal_augmentation_"
        "ratio_1_seed_42"
    )
)

BASELINE_OUTPUT_DIR = Path(
    "../outputs/baseline_classification"
)

AUGMENTED_OUTPUT_DIR = Path(
    "../outputs/"
    "synthetic_augmentation_classification"
)

MULTISEED_OUTPUT_DIR = Path(
    "../outputs/"
    "multiseed_augmentation_robustness"
)

MULTISEED_CHECKPOINT_DIR = Path(
    "../checkpoints/"
    "multiseed_augmentation_robustness"
)

REFERENCE_BASELINE_DIR = (
    BASELINE_OUTPUT_DIR
    / "10_percent_seed_42"
)

REFERENCE_AUGMENTED_DIR = (
    AUGMENTED_OUTPUT_DIR
    / (
        "10_percent_real_plus_"
        "synthetic_1_to_1_seed_42"
    )
)

required_source_files = [
    OFFICIAL_DATA_DIR / "X_train.npy",
    OFFICIAL_DATA_DIR / "y_train.npy",
    OFFICIAL_DATA_DIR / "X_validation.npy",
    OFFICIAL_DATA_DIR / "y_validation.npy",
    OFFICIAL_DATA_DIR / "X_test.npy",
    OFFICIAL_DATA_DIR / "y_test.npy",

    LIMITED_DATA_DIR
    / "indices_10_percent.npy",

    SYNTHETIC_DATASET_DIR
    / "X_synthetic.npy",

    SYNTHETIC_DATASET_DIR
    / "y_synthetic.npy",

    REFERENCE_BASELINE_DIR
    / "test_metrics.csv",

    REFERENCE_AUGMENTED_DIR
    / "test_metrics.csv"
]

missing_source_files = [
    path
    for path in required_source_files
    if not path.exists()
]

if missing_source_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(
            str(path.resolve())
            for path in missing_source_files
        )
    )

MULTISEED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MULTISEED_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "All required source files were found."
)

print(
    "Multi-seed output directory:",
    MULTISEED_OUTPUT_DIR.resolve()
)

print(
    "Multi-seed checkpoint directory:",
    MULTISEED_CHECKPOINT_DIR.resolve()
)

In [ ]:
run_status_records = []

for configuration_name in CONFIGURATIONS:
    for seed in NEW_TRAINING_SEEDS:
        run_name = (
            f"{configuration_name}_"
            f"seed_{seed}"
        )

        run_output_directory = (
            MULTISEED_OUTPUT_DIR
            / run_name
        )

        checkpoint_path = (
            MULTISEED_CHECKPOINT_DIR
            / f"{run_name}.keras"
        )

        history_path = (
            run_output_directory
            / "training_history.csv"
        )

        validation_path = (
            run_output_directory
            / "validation_results.csv"
        )

        test_path = (
            run_output_directory
            / "test_metrics.csv"
        )

        checkpoint_exists = (
            checkpoint_path.exists()
        )

        history_exists = (
            history_path.exists()
        )

        validation_exists = (
            validation_path.exists()
        )

        test_exists = (
            test_path.exists()
        )

        training_complete = (
            checkpoint_exists
            and history_exists
        )

        evaluation_complete = (
            validation_exists
            and test_exists
        )

        run_complete = (
            training_complete
            and evaluation_complete
        )

        inconsistent_artifacts = (
            checkpoint_exists
            != history_exists
        ) or (
            evaluation_complete
            and not training_complete
        ) or (
            validation_exists
            != test_exists
        )

        if inconsistent_artifacts:
            status = (
                "inconsistent_artifacts"
            )
        elif run_complete:
            status = "complete"
        elif training_complete:
            status = "trained_awaiting_evaluation"
        else:
            status = "not_started"

        run_status_records.append({
            "configuration":
                configuration_name,

            "seed":
                seed,

            "status":
                status,

            "checkpoint_exists":
                checkpoint_exists,

            "history_exists":
                history_exists,

            "validation_exists":
                validation_exists,

            "test_exists":
                test_exists,

            "training_complete":
                training_complete,

            "run_complete":
                run_complete,

            "output_directory":
                str(
                    run_output_directory
                )
        })

run_status_df = pd.DataFrame(
    run_status_records
)

display(
    run_status_df
)

inconsistent_runs = run_status_df[
    run_status_df["status"]
    == "inconsistent_artifacts"
]

if not inconsistent_runs.empty:
    raise RuntimeError(
        "Inconsistent experiment artifacts "
        "were detected. Inspect these runs "
        "before continuing:\n"
        + inconsistent_runs[
            [
                "configuration",
                "seed",
                "status"
            ]
        ].to_string(index=False)
    )

if RUN_NEW_TRAINING:
    runs_to_train = run_status_df[
        run_status_df["status"]
        == "not_started"
    ]

    print(
        "Runs authorized for training:",
        len(runs_to_train)
    )

else:
    print(
        "SAFE INSPECTION MODE: no new "
        "model training is authorized."
    )

## 2. Fixed Data Sources

All training seeds use exactly the same real and synthetic observations. This experiment measures training stochasticity rather than variability in data selection or synthetic generation.

In [ ]:
X_train_complete = np.load(
    OFFICIAL_DATA_DIR / "X_train.npy",
    mmap_mode="r"
)

y_train_complete = np.load(
    OFFICIAL_DATA_DIR / "y_train.npy",
    mmap_mode="r"
)

source_indices = np.load(
    LIMITED_DATA_DIR
    / "indices_10_percent.npy"
)

X_real = np.asarray(
    X_train_complete[source_indices],
    dtype=np.float32
)

y_real = np.asarray(
    y_train_complete[source_indices],
    dtype=np.uint8
)

X_synthetic = np.asarray(
    np.load(
        SYNTHETIC_DATASET_DIR
        / "X_synthetic.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_synthetic = np.asarray(
    np.load(
        SYNTHETIC_DATASET_DIR
        / "y_synthetic.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

X_validation = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "X_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_validation = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "y_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

X_test = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "X_test.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_test = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "y_test.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

print("Real tensor:", X_real.shape)
print("Synthetic tensor:", X_synthetic.shape)
print(
    "Validation tensor:",
    X_validation.shape
)
print("Test tensor:", X_test.shape)

print(
    "Real class counts:",
    np.bincount(
        y_real,
        minlength=2
    )
)

print(
    "Synthetic class counts:",
    np.bincount(
        y_synthetic,
        minlength=2
    )
)

In [ ]:
assert X_real.shape == (
    1150,
    5,
    150
)

assert y_real.shape == (
    1150,
)

assert X_synthetic.shape == (
    1150,
    5,
    150
)

assert y_synthetic.shape == (
    1150,
)

assert X_validation.shape == (
    6988,
    5,
    150
)

assert y_validation.shape == (
    6988,
)

assert X_test.shape == (
    6996,
    5,
    150
)

assert y_test.shape == (
    6996,
)

assert np.array_equal(
    np.bincount(
        y_real,
        minlength=2
    ),
    [575, 575]
)

assert np.array_equal(
    np.bincount(
        y_synthetic,
        minlength=2
    ),
    [575, 575]
)

for dataset_name, dataset in {
    "X_real": X_real,
    "X_synthetic": X_synthetic,
    "X_validation": X_validation,
    "X_test": X_test
}.items():
    assert dataset.dtype == np.float32
    assert np.isfinite(dataset).all()
    assert dataset.min() >= 0.0
    assert dataset.max() <= 1.0

print(
    "All fixed datasets passed "
    "structural validation."
)

In [ ]:
training_arrays = {
    "real_only": (
        X_real,
        y_real
    ),

    "real_plus_synthetic": (
        np.concatenate(
            [
                X_real,
                X_synthetic
            ],
            axis=0
        ),
        np.concatenate(
            [
                y_real,
                y_synthetic
            ],
            axis=0
        )
    )
}

for (
    configuration_name,
    (
        configuration_features,
        configuration_labels
    )
) in training_arrays.items():
    print(
        configuration_name,
        "tensor:",
        configuration_features.shape,
        "class counts:",
        np.bincount(
            configuration_labels,
            minlength=2
        )
    )

assert (
    training_arrays[
        "real_only"
    ][0].shape
    == (1150, 5, 150)
)

assert (
    training_arrays[
        "real_plus_synthetic"
    ][0].shape
    == (2300, 5, 150)
)

assert np.array_equal(
    np.bincount(
        training_arrays[
            "real_plus_synthetic"
        ][1],
        minlength=2
    ),
    [1150, 1150]
)

In [ ]:
X_validation_model = (
    X_validation[..., np.newaxis]
)

X_test_model = (
    X_test[..., np.newaxis]
)

def build_datasets(
    configuration_name,
    training_seed
):
    X_configuration, y_configuration = (
        training_arrays[
            configuration_name
        ]
    )

    X_configuration_model = (
        X_configuration[
            ...,
            np.newaxis
        ]
    )

    training_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_configuration_model,
            y_configuration
        ))
        .shuffle(
            buffer_size=len(
                y_configuration
            ),
            seed=training_seed,
            reshuffle_each_iteration=True
        )
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    validation_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_validation_model,
            y_validation
        ))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    test_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_test_model,
            y_test
        ))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    return (
        training_dataset,
        validation_dataset,
        test_dataset
    )


for configuration_name in CONFIGURATIONS:
    (
        example_training_dataset,
        example_validation_dataset,
        example_test_dataset
    ) = build_datasets(
        configuration_name,
        REFERENCE_SEED
    )

    print(
        configuration_name,
        "training batches:",
        len(example_training_dataset)
    )

print(
    "Validation batches:",
    len(example_validation_dataset)
)

print(
    "Test batches:",
    len(example_test_dataset)
)

## 3. Fixed CNN Architecture

Every run uses the same 29,121-parameter CNN as the real-only learning-curve and seed-42 augmented experiments. This controls model capacity across configurations and seeds.

In [ ]:
def build_fixed_cnn(
    input_shape=(5, 150, 1)
):
    model = models.Sequential([
        layers.Input(
            shape=input_shape
        ),

        layers.Conv2D(
            filters=16,
            kernel_size=(3, 7),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            filters=32,
            kernel_size=(3, 5),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            filters=64,
            kernel_size=(3, 3),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            units=32,
            activation="relu",
            kernel_regularizer=(
                regularizers.l2(1e-4)
            )
        ),

        layers.Dropout(0.30),

        layers.Dense(
            units=1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),

        loss="binary_crossentropy",

        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.AUC(
                name="roc_auc"
            ),

            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            ),

            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            )
        ]
    )

    return model

In [ ]:
tf.keras.backend.clear_session()

verification_model = build_fixed_cnn()

total_parameters = (
    verification_model.count_params()
)

trainable_parameters = int(
    np.sum([
        np.prod(variable.shape)
        for variable
        in verification_model.trainable_weights
    ])
)

non_trainable_parameters = int(
    np.sum([
        np.prod(variable.shape)
        for variable
        in verification_model.non_trainable_weights
    ])
)

assert total_parameters == 29121
assert trainable_parameters == 28897
assert non_trainable_parameters == 224

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)

print(
    "Non-trainable parameters:",
    non_trainable_parameters
)

print(
    "Architecture verified against "
    "the previous experiments."
)

verification_model.summary()

del verification_model

tf.keras.backend.clear_session()

## 4. Protected Multi-Seed Training

Each run has an independent checkpoint, history file, training log, and configuration record. Existing completed training artifacts are loaded or skipped and are never overwritten.

In [ ]:
def set_training_seed(training_seed):
    tf.keras.backend.clear_session()

    random.seed(training_seed)
    np.random.seed(training_seed)
    tf.random.set_seed(training_seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

In [ ]:
def get_run_paths(
    configuration_name,
    training_seed
):
    run_name = (
        f"{configuration_name}_"
        f"seed_{training_seed}"
    )

    output_directory = (
        MULTISEED_OUTPUT_DIR
        / run_name
    )

    checkpoint_path = (
        MULTISEED_CHECKPOINT_DIR
        / f"{run_name}.keras"
    )

    return {
        "run_name":
            run_name,

        "output_directory":
            output_directory,

        "checkpoint":
            checkpoint_path,

        "history":
            output_directory
            / "training_history.csv",

        "training_log":
            output_directory
            / "training_log.csv",

        "configuration":
            output_directory
            / "run_configuration.json",

        "validation_results":
            output_directory
            / "validation_results.csv",

        "test_metrics":
            output_directory
            / "test_metrics.csv"
    }

In [ ]:
def train_single_run(
    configuration_name,
    training_seed
):
    run_paths = get_run_paths(
        configuration_name,
        training_seed
    )

    checkpoint_exists = (
        run_paths["checkpoint"].exists()
    )

    history_exists = (
        run_paths["history"].exists()
    )

    if (
        checkpoint_exists
        and history_exists
    ):
        print(
            "Training already complete; "
            "skipping:",
            run_paths["run_name"]
        )

        return {
            "configuration":
                configuration_name,

            "seed":
                training_seed,

            "status":
                "already_trained"
        }

    if (
        checkpoint_exists
        != history_exists
    ):
        raise RuntimeError(
            "Inconsistent training artifacts "
            "for "
            + run_paths["run_name"]
        )

    protected_existing_files = [
        path
        for path in [
            run_paths[
                "validation_results"
            ],
            run_paths[
                "test_metrics"
            ],
            run_paths[
                "training_log"
            ],
            run_paths[
                "configuration"
            ]
        ]
        if path.exists()
    ]

    if protected_existing_files:
        raise FileExistsError(
            "Unexpected existing files for "
            + run_paths["run_name"]
            + ":\n"
            + "\n".join(
                str(path.resolve())
                for path
                in protected_existing_files
            )
        )

    run_paths[
        "output_directory"
    ].mkdir(
        parents=True,
        exist_ok=True
    )

    set_training_seed(
        training_seed
    )

    (
        training_dataset,
        validation_dataset,
        _
    ) = build_datasets(
        configuration_name,
        training_seed
    )

    model = build_fixed_cnn()

    assert model.count_params() == 29121

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=run_paths[
                "checkpoint"
            ],
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),

        tf.keras.callbacks.CSVLogger(
            run_paths[
                "training_log"
            ]
        )
    ]

    print()
    print("=" * 70)
    print(
        "Training:",
        run_paths["run_name"]
    )
    print(
        "Training observations:",
        len(
            training_arrays[
                configuration_name
            ][1]
        )
    )
    print("=" * 70)

    history = model.fit(
        training_dataset,
        validation_data=(
            validation_dataset
        ),
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        shuffle=False,
        verbose=1
    )

    history_df = pd.DataFrame(
        history.history
    )

    history_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(history_df) + 1
        )
    )

    history_df.to_csv(
        run_paths["history"],
        index=False
    )

    best_epoch_index = (
        history_df["val_loss"].idxmin()
    )

    best_epoch = int(
        history_df.loc[
            best_epoch_index,
            "epoch"
        ]
    )

    best_validation_loss = float(
        history_df.loc[
            best_epoch_index,
            "val_loss"
        ]
    )

    run_configuration = {
        "run_name":
            run_paths["run_name"],

        "configuration":
            configuration_name,

        "configuration_display_name":
            CONFIGURATIONS[
                configuration_name
            ]["display_name"],

        "training_seed":
            training_seed,

        "real_subset":
            REAL_SUBSET,

        "synthetic_ratio":
            (
                SYNTHETIC_RATIO
                if CONFIGURATIONS[
                    configuration_name
                ]["use_synthetic"]
                else 0.0
            ),

        "training_observations":
            int(
                len(
                    training_arrays[
                        configuration_name
                    ][1]
                )
            ),

        "batch_size":
            BATCH_SIZE,

        "maximum_epochs":
            MAX_EPOCHS,

        "completed_epochs":
            int(
                len(history_df)
            ),

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss,

        "model_parameters":
            int(
                model.count_params()
            )
    }

    with open(
        run_paths["configuration"],
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            run_configuration,
            file,
            indent=2
        )

    assert (
        run_paths["checkpoint"].exists()
    )

    assert (
        run_paths["history"].exists()
    )

    print(
        "Completed:",
        run_paths["run_name"]
    )

    print(
        "Epochs:",
        len(history_df)
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best validation loss:",
        best_validation_loss
    )

    del model
    del history

    tf.keras.backend.clear_session()

    return {
        "configuration":
            configuration_name,

        "seed":
            training_seed,

        "status":
            "trained",

        "completed_epochs":
            int(
                len(history_df)
            ),

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss
    }

In [ ]:
pending_training_records = []

for training_seed in NEW_TRAINING_SEEDS:
    for configuration_name in CONFIGURATIONS:
        run_paths = get_run_paths(
            configuration_name,
            training_seed
        )

        training_complete = (
            run_paths["checkpoint"].exists()
            and run_paths["history"].exists()
        )

        pending_training_records.append({
            "configuration":
                configuration_name,

            "seed":
                training_seed,

            "training_observations":
                len(
                    training_arrays[
                        configuration_name
                    ][1]
                ),

            "training_complete":
                training_complete,

            "action":
                (
                    "skip"
                    if training_complete
                    else "train"
                )
        })

pending_training_df = pd.DataFrame(
    pending_training_records
)

display(
    pending_training_df
)

print(
    "Runs still requiring training:",
    int(
        (
            pending_training_df[
                "action"
            ] == "train"
        ).sum()
    )
)

In [ ]:
if not RUN_NEW_TRAINING:
    print(
        "Training remains disabled. "
        "Set RUN_NEW_TRAINING = True "
        "only when ready."
    )

else:
    training_run_records = []

    for training_seed in (
        NEW_TRAINING_SEEDS
    ):
        for configuration_name in (
            CONFIGURATIONS
        ):
            run_result = (
                train_single_run(
                    configuration_name,
                    training_seed
                )
            )

            training_run_records.append(
                run_result
            )

    training_run_summary_df = (
        pd.DataFrame(
            training_run_records
        )
    )

    display(
        training_run_summary_df
    )

    RUN_NEW_TRAINING = False

    print(
        "All requested training calls "
        "finished."
    )

    print(
        "RUN_NEW_TRAINING was reset "
        "to False in memory."
    )

## 5. Validation-Threshold Selection and Test Evaluation

For each trained run, the classification threshold is selected exclusively from the validation partition by maximizing macro-F1, with balanced accuracy as the tie-breaker. The selected threshold is then locked before evaluating the official test partition.

In [ ]:
def calculate_binary_metrics(
    true_labels,
    predicted_labels,
    probabilities
):
    return {
        "accuracy":
            accuracy_score(
                true_labels,
                predicted_labels
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_labels,
                predicted_labels
            ),

        "macro_f1":
            f1_score(
                true_labels,
                predicted_labels,
                average="macro",
                zero_division=0
            ),

        "bird_precision":
            precision_score(
                true_labels,
                predicted_labels,
                pos_label=0,
                zero_division=0
            ),

        "bird_recall":
            recall_score(
                true_labels,
                predicted_labels,
                pos_label=0,
                zero_division=0
            ),

        "bird_f1":
            f1_score(
                true_labels,
                predicted_labels,
                pos_label=0,
                zero_division=0
            ),

        "drone_precision":
            precision_score(
                true_labels,
                predicted_labels,
                pos_label=1,
                zero_division=0
            ),

        "drone_recall":
            recall_score(
                true_labels,
                predicted_labels,
                pos_label=1,
                zero_division=0
            ),

        "drone_f1":
            f1_score(
                true_labels,
                predicted_labels,
                pos_label=1,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                true_labels,
                probabilities
            )
    }

In [ ]:
def evaluate_single_run(
    configuration_name,
    training_seed
):
    run_paths = get_run_paths(
        configuration_name,
        training_seed
    )

    threshold_search_path = (
        run_paths["output_directory"]
        / "validation_threshold_search.csv"
    )

    predictions_path = (
        run_paths["output_directory"]
        / "test_predictions.csv"
    )

    required_training_files = [
        run_paths["checkpoint"],
        run_paths["history"],
        run_paths["configuration"]
    ]

    missing_training_files = [
        path
        for path in required_training_files
        if not path.exists()
    ]

    if missing_training_files:
        raise FileNotFoundError(
            "Missing training artifacts for "
            + run_paths["run_name"]
            + ":\n"
            + "\n".join(
                str(path.resolve())
                for path
                in missing_training_files
            )
        )

    evaluation_files = [
        run_paths["validation_results"],
        run_paths["test_metrics"],
        threshold_search_path,
        predictions_path
    ]

    existing_evaluation_files = [
        path.exists()
        for path in evaluation_files
    ]

    if all(existing_evaluation_files):
        existing_test_metrics = pd.read_csv(
            run_paths["test_metrics"]
        )

        print(
            "Evaluation already complete; "
            "loading:",
            run_paths["run_name"]
        )

        return existing_test_metrics.iloc[
            0
        ].to_dict()

    if any(existing_evaluation_files):
        raise RuntimeError(
            "Incomplete evaluation artifacts "
            "were found for "
            + run_paths["run_name"]
            + ". Inspect this run before "
              "continuing."
        )

    set_training_seed(
        training_seed
    )

    (
        _,
        validation_dataset,
        test_dataset
    ) = build_datasets(
        configuration_name,
        training_seed
    )

    model = tf.keras.models.load_model(
        run_paths["checkpoint"]
    )

    assert model.count_params() == 29121

    history_df = pd.read_csv(
        run_paths["history"]
    )

    best_epoch_index = (
        history_df["val_loss"].idxmin()
    )

    best_epoch = int(
        history_df.loc[
            best_epoch_index,
            "epoch"
        ]
    )

    best_validation_loss = float(
        history_df.loc[
            best_epoch_index,
            "val_loss"
        ]
    )

    validation_probabilities = (
        model.predict(
            validation_dataset,
            verbose=0
        )
        .reshape(-1)
    )

    assert (
        validation_probabilities.shape
        == y_validation.shape
    )

    threshold_records = []

    for threshold in np.linspace(
        0.01,
        0.99,
        199
    ):
        validation_predictions = (
            validation_probabilities
            >= threshold
        ).astype(np.uint8)

        threshold_records.append({
            "threshold":
                float(threshold),

            "accuracy":
                accuracy_score(
                    y_validation,
                    validation_predictions
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_validation,
                    validation_predictions
                ),

            "macro_f1":
                f1_score(
                    y_validation,
                    validation_predictions,
                    average="macro",
                    zero_division=0
                ),

            "bird_recall":
                recall_score(
                    y_validation,
                    validation_predictions,
                    pos_label=0,
                    zero_division=0
                ),

            "drone_recall":
                recall_score(
                    y_validation,
                    validation_predictions,
                    pos_label=1,
                    zero_division=0
                )
        })

    threshold_search_df = pd.DataFrame(
        threshold_records
    )

    optimal_threshold_row = (
        threshold_search_df
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "accuracy"
            ],
            ascending=[
                False,
                False,
                False
            ],
            kind="mergesort"
        )
        .iloc[0]
    )

    locked_threshold = float(
        optimal_threshold_row[
            "threshold"
        ]
    )

    validation_results_df = (
        pd.DataFrame([{
            "configuration":
                configuration_name,

            "configuration_display_name":
                CONFIGURATIONS[
                    configuration_name
                ]["display_name"],

            "seed":
                training_seed,

            "best_epoch":
                best_epoch,

            "best_validation_loss":
                best_validation_loss,

            "threshold":
                locked_threshold,

            "validation_accuracy":
                float(
                    optimal_threshold_row[
                        "accuracy"
                    ]
                ),

            "validation_balanced_accuracy":
                float(
                    optimal_threshold_row[
                        "balanced_accuracy"
                    ]
                ),

            "validation_macro_f1":
                float(
                    optimal_threshold_row[
                        "macro_f1"
                    ]
                ),

            "validation_bird_recall":
                float(
                    optimal_threshold_row[
                        "bird_recall"
                    ]
                ),

            "validation_drone_recall":
                float(
                    optimal_threshold_row[
                        "drone_recall"
                    ]
                )
        }])
    )

    # Threshold is now locked before test inference.
    test_probabilities = (
        model.predict(
            test_dataset,
            verbose=0
        )
        .reshape(-1)
    )

    assert (
        test_probabilities.shape
        == y_test.shape
    )

    assert np.isfinite(
        test_probabilities
    ).all()

    test_predictions = (
        test_probabilities
        >= locked_threshold
    ).astype(np.uint8)

    test_metric_values = (
        calculate_binary_metrics(
            y_test,
            test_predictions,
            test_probabilities
        )
    )

    test_metrics_record = {
        "configuration":
            configuration_name,

        "configuration_display_name":
            CONFIGURATIONS[
                configuration_name
            ]["display_name"],

        "seed":
            training_seed,

        "training_observations":
            int(
                len(
                    training_arrays[
                        configuration_name
                    ][1]
                )
            ),

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss,

        "threshold":
            locked_threshold,

        **test_metric_values
    }

    test_metrics_df = pd.DataFrame([
        test_metrics_record
    ])

    test_predictions_df = pd.DataFrame({
        "true_binary_label":
            y_test,

        "drone_probability":
            test_probabilities,

        "predicted_binary_label":
            test_predictions,

        "correct":
            (
                y_test
                == test_predictions
            )
    })

    threshold_search_df.to_csv(
        threshold_search_path,
        index=False
    )

    validation_results_df.to_csv(
        run_paths[
            "validation_results"
        ],
        index=False
    )

    test_metrics_df.to_csv(
        run_paths["test_metrics"],
        index=False
    )

    test_predictions_df.to_csv(
        predictions_path,
        index=False
    )

    print()
    print(
        "Evaluated:",
        run_paths["run_name"]
    )

    print(
        "Locked threshold:",
        round(
            locked_threshold,
            4
        )
    )

    print(
        "Test macro-F1:",
        round(
            test_metric_values[
                "macro_f1"
            ],
            4
        )
    )

    print(
        "Test balanced accuracy:",
        round(
            test_metric_values[
                "balanced_accuracy"
            ],
            4
        )
    )

    del model

    tf.keras.backend.clear_session()

    return test_metrics_record

In [ ]:
assert RUN_NEW_TRAINING is False

new_run_test_records = []

for training_seed in NEW_TRAINING_SEEDS:
    for configuration_name in CONFIGURATIONS:
        evaluation_result = (
            evaluate_single_run(
                configuration_name,
                training_seed
            )
        )

        new_run_test_records.append(
            evaluation_result
        )

new_run_test_metrics_df = (
    pd.DataFrame(
        new_run_test_records
    )
    .sort_values([
        "seed",
        "configuration"
    ])
    .reset_index(drop=True)
)

new_run_test_metrics_df.to_csv(
    MULTISEED_OUTPUT_DIR
    / "new_seed_test_metrics.csv",
    index=False
)

display(
    new_run_test_metrics_df[
        [
            "configuration_display_name",
            "seed",
            "best_epoch",
            "best_validation_loss",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "bird_precision",
            "bird_recall",
            "bird_f1",
            "drone_recall",
            "roc_auc"
        ]
    ].style.format({
        "best_validation_loss": "{:.4f}",
        "threshold": "{:.4f}",
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "bird_precision": "{:.4f}",
        "bird_recall": "{:.4f}",
        "bird_f1": "{:.4f}",
        "drone_recall": "{:.4f}",
        "roc_auc": "{:.4f}"
    })
)

## 6. Complete Five-Seed Results

The completed seed-42 metrics from the real-only and augmented experiments are combined with the four new seeds. Seed 42 is loaded from the original experiment artifacts rather than retrained.

In [ ]:
reference_real_df = pd.read_csv(
    REFERENCE_BASELINE_DIR
    / "test_metrics.csv"
)

reference_augmented_df = pd.read_csv(
    REFERENCE_AUGMENTED_DIR
    / "test_metrics.csv"
)

required_metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_precision",
    "drone_recall",
    "drone_f1",
    "roc_auc"
]

def extract_reference_training_metadata(
    source_dataframe,
    experiment_directory
):
    source_row = (
        source_dataframe.iloc[0]
    )

    if (
        "best_epoch"
        in source_dataframe.columns
        and "best_validation_loss"
        in source_dataframe.columns
    ):
        best_epoch = int(
            source_row["best_epoch"]
        )

        best_validation_loss = float(
            source_row[
                "best_validation_loss"
            ]
        )

    else:
        history_path = (
            experiment_directory
            / "training_history.csv"
        )

        if not history_path.exists():
            raise FileNotFoundError(
                "Training metadata are absent "
                "from test_metrics.csv and the "
                "training history was not found: "
                + str(
                    history_path.resolve()
                )
            )

        history_df = pd.read_csv(
            history_path
        )

        if "epoch" not in history_df.columns:
            history_df.insert(
                0,
                "epoch",
                np.arange(
                    1,
                    len(history_df) + 1
                )
            )

        best_history_index = (
            history_df[
                "val_loss"
            ].idxmin()
        )

        best_epoch = int(
            history_df.loc[
                best_history_index,
                "epoch"
            ]
        )

        best_validation_loss = float(
            history_df.loc[
                best_history_index,
                "val_loss"
            ]
        )

    return (
        best_epoch,
        best_validation_loss
    )


reference_records = []

reference_sources = [
    {
        "configuration":
            "real_only",

        "configuration_display_name":
            "10% real-only",

        "directory":
            REFERENCE_BASELINE_DIR,

        "metrics":
            reference_real_df,

        "training_observations":
            1150
    },

    {
        "configuration":
            "real_plus_synthetic",

        "configuration_display_name":
            "10% real + synthetic 1:1",

        "directory":
            REFERENCE_AUGMENTED_DIR,

        "metrics":
            reference_augmented_df,

        "training_observations":
            2300
    }
]

for reference_source in reference_sources:
    source_dataframe = (
        reference_source["metrics"]
    )

    source_row = (
        source_dataframe.iloc[0]
    )

    missing_metric_columns = [
        metric_column
        for metric_column
        in required_metric_columns
        if metric_column
        not in source_dataframe.columns
    ]

    if missing_metric_columns:
        raise KeyError(
            "Missing seed-42 metric columns: "
            + ", ".join(
                missing_metric_columns
            )
        )

    (
        best_epoch,
        best_validation_loss
    ) = extract_reference_training_metadata(
        source_dataframe,
        reference_source[
            "directory"
        ]
    )

    if (
        "threshold"
        not in source_dataframe.columns
    ):
        raise KeyError(
            "The seed-42 metrics file does "
            "not contain the locked threshold."
        )

    reference_record = {
        "configuration":
            reference_source[
                "configuration"
            ],

        "configuration_display_name":
            reference_source[
                "configuration_display_name"
            ],

        "seed":
            REFERENCE_SEED,

        "training_observations":
            reference_source[
                "training_observations"
            ],

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss,

        "threshold":
            float(
                source_row["threshold"]
            )
    }

    for metric_column in (
        required_metric_columns
    ):
        reference_record[
            metric_column
        ] = float(
            source_row[
                metric_column
            ]
        )

    reference_records.append(
        reference_record
    )

reference_metrics_df = pd.DataFrame(
    reference_records
)

display(
    reference_metrics_df.style.format({
        "best_validation_loss": "{:.4f}",
        "threshold": "{:.4f}",
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "bird_precision": "{:.4f}",
        "bird_recall": "{:.4f}",
        "bird_f1": "{:.4f}",
        "drone_precision": "{:.4f}",
        "drone_recall": "{:.4f}",
        "drone_f1": "{:.4f}",
        "roc_auc": "{:.4f}"
    })
)

In [ ]:
all_seed_metrics_df = pd.concat(
    [
        reference_metrics_df,
        new_run_test_metrics_df
    ],
    ignore_index=True
)

all_seed_metrics_df = (
    all_seed_metrics_df
    .sort_values([
        "seed",
        "configuration"
    ])
    .reset_index(drop=True)
)

assert len(
    all_seed_metrics_df
) == 10

assert set(
    all_seed_metrics_df["seed"]
) == set(
    EXPERIMENT_SEEDS
)

assert (
    all_seed_metrics_df
    .groupby("seed")
    .size()
    .eq(2)
    .all()
)

assert not (
    all_seed_metrics_df
    .duplicated([
        "configuration",
        "seed"
    ])
    .any()
)

all_seed_metrics_df.to_csv(
    MULTISEED_OUTPUT_DIR
    / "all_seed_test_metrics.csv",
    index=False
)

display(
    all_seed_metrics_df[
        [
            "configuration_display_name",
            "seed",
            "best_epoch",
            "best_validation_loss",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "bird_precision",
            "bird_recall",
            "bird_f1",
            "drone_recall",
            "roc_auc"
        ]
    ].style.format({
        "best_validation_loss": "{:.4f}",
        "threshold": "{:.4f}",
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "bird_precision": "{:.4f}",
        "bird_recall": "{:.4f}",
        "bird_f1": "{:.4f}",
        "drone_recall": "{:.4f}",
        "roc_auc": "{:.4f}"
    })
)

In [ ]:
summary_metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

configuration_summary_df = (
    all_seed_metrics_df
    .groupby(
        [
            "configuration",
            "configuration_display_name"
        ],
        observed=True
    )[summary_metric_columns]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
)

configuration_summary_df.columns = [
    f"{metric}_{statistic}"
    for metric, statistic
    in configuration_summary_df.columns
]

configuration_summary_df = (
    configuration_summary_df
    .reset_index()
)

configuration_summary_df.to_csv(
    MULTISEED_OUTPUT_DIR
    / "configuration_summary.csv",
    index=False
)

display(
    configuration_summary_df.style.format({
        column: "{:.4f}"
        for column
        in configuration_summary_df.columns
        if column not in [
            "configuration",
            "configuration_display_name"
        ]
    })
)

### Paired seed-level improvements

Each augmented run is compared with the real-only run using the same training seed. The reported t-based intervals summarize uncertainty in the five paired differences and are interpreted descriptively.


In [ ]:
paired_improvement_records = []

paired_seed_records = []

for metric_name in (
    summary_metric_columns
):
    paired_metric_df = (
        all_seed_metrics_df
        .pivot(
            index="seed",
            columns="configuration",
            values=metric_name
        )
        .reindex(EXPERIMENT_SEEDS)
    )

    paired_metric_df[
        "improvement"
    ] = (
        paired_metric_df[
            "real_plus_synthetic"
        ]
        - paired_metric_df[
            "real_only"
        ]
    )

    for (
        training_seed,
        seed_row
    ) in paired_metric_df.iterrows():
        paired_seed_records.append({
            "seed":
                int(training_seed),

            "metric":
                metric_name,

            "real_only":
                float(
                    seed_row[
                        "real_only"
                    ]
                ),

            "real_plus_synthetic":
                float(
                    seed_row[
                        "real_plus_synthetic"
                    ]
                ),

            "improvement":
                float(
                    seed_row[
                        "improvement"
                    ]
                )
        })

    improvements = (
        paired_metric_df[
            "improvement"
        ].to_numpy()
    )

    number_of_seeds = len(
        improvements
    )

    mean_improvement = float(
        improvements.mean()
    )

    improvement_std = float(
        improvements.std(
            ddof=1
        )
    )

    standard_error = (
        improvement_std
        / np.sqrt(number_of_seeds)
    )

    critical_t = float(
        stats.t.ppf(
            0.975,
            df=number_of_seeds - 1
        )
    )

    confidence_margin = (
        critical_t
        * standard_error
    )

    paired_improvement_records.append({
        "metric":
            metric_name,

        "seeds":
            number_of_seeds,

        "real_only_mean":
            float(
                paired_metric_df[
                    "real_only"
                ].mean()
            ),

        "augmented_mean":
            float(
                paired_metric_df[
                    "real_plus_synthetic"
                ].mean()
            ),

        "mean_paired_improvement":
            mean_improvement,

        "improvement_standard_deviation":
            improvement_std,

        "confidence_interval_95_lower":
            mean_improvement
            - confidence_margin,

        "confidence_interval_95_upper":
            mean_improvement
            + confidence_margin,

        "augmented_wins":
            int(
                np.sum(
                    improvements > 0
                )
            ),

        "ties":
            int(
                np.sum(
                    np.isclose(
                        improvements,
                        0.0
                    )
                )
            ),

        "augmented_losses":
            int(
                np.sum(
                    improvements < 0
                )
            )
    })

paired_seed_metrics_df = pd.DataFrame(
    paired_seed_records
)

paired_improvement_summary_df = (
    pd.DataFrame(
        paired_improvement_records
    )
)

paired_seed_metrics_df.to_csv(
    MULTISEED_OUTPUT_DIR
    / "paired_seed_metrics.csv",
    index=False
)

paired_improvement_summary_df.to_csv(
    MULTISEED_OUTPUT_DIR
    / "paired_improvement_summary.csv",
    index=False
)

display(
    paired_improvement_summary_df.style.format({
        "real_only_mean": "{:.4f}",
        "augmented_mean": "{:.4f}",
        "mean_paired_improvement": "{:+.4f}",
        "improvement_standard_deviation": "{:.4f}",
        "confidence_interval_95_lower":
            "{:+.4f}",
        "confidence_interval_95_upper":
            "{:+.4f}"
    })
)

### Paired performance visualization

Lines connect models trained with the same seed, making both the direction of the augmentation effect and seed sensitivity visible.


In [ ]:
plot_metrics = {
    "balanced_accuracy":
        "Balanced Accuracy",

    "macro_f1":
        "Macro-F1",

    "bird_f1":
        "Bird F1",

    "roc_auc":
        "ROC-AUC"
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 10),
    constrained_layout=True
)

configuration_x_positions = {
    "real_only": 0,
    "real_plus_synthetic": 1
}

configuration_x_labels = [
    "10% real-only",
    "10% real +\nsynthetic 1:1"
]

for axis, (
    metric_name,
    metric_title
) in zip(
    axes.flat,
    plot_metrics.items()
):
    metric_data = (
        all_seed_metrics_df[
            [
                "configuration",
                "seed",
                metric_name
            ]
        ]
    )

    for training_seed in (
        EXPERIMENT_SEEDS
    ):
        seed_data = (
            metric_data[
                metric_data["seed"]
                == training_seed
            ]
            .set_index(
                "configuration"
            )
        )

        x_values = [
            configuration_x_positions[
                "real_only"
            ],
            configuration_x_positions[
                "real_plus_synthetic"
            ]
        ]

        y_values = [
            seed_data.loc[
                "real_only",
                metric_name
            ],
            seed_data.loc[
                "real_plus_synthetic",
                metric_name
            ]
        ]

        axis.plot(
            x_values,
            y_values,
            marker="o",
            linewidth=1.8,
            alpha=0.80,
            label=f"Seed {training_seed}"
        )

    axis.set_xticks([
        0,
        1
    ])

    axis.set_xticklabels(
        configuration_x_labels
    )

    axis.set_ylim(
        0.0,
        1.02
    )

    axis.set_ylabel("Test score")
    axis.set_title(metric_title)
    axis.grid(
        alpha=0.25
    )

axes[0, 0].legend(
    title="Training seed",
    loc="lower right"
)

fig.suptitle(
    "Paired Multi-Seed Performance: "
    "Real-Only versus Synthetic-Augmented Training",
    fontsize=14
)

plt.show()

## 7. Stability and Worst-Case Performance

In addition to mean performance, this section examines whether augmentation reduces sensitivity to model initialization and improves the worst observed outcome across training seeds.

### Variability and worst-case outcomes

The comparison includes dispersion and the weakest observed result, since low-data models may be sensitive to initialization and training order.


In [ ]:
stability_metrics = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

stability_records = []

for metric_name in stability_metrics:
    real_values = (
        all_seed_metrics_df.loc[
            all_seed_metrics_df[
                "configuration"
            ] == "real_only",
            metric_name
        ]
        .to_numpy()
    )

    augmented_values = (
        all_seed_metrics_df.loc[
            all_seed_metrics_df[
                "configuration"
            ] == "real_plus_synthetic",
            metric_name
        ]
        .to_numpy()
    )

    real_std = float(
        real_values.std(ddof=1)
    )

    augmented_std = float(
        augmented_values.std(ddof=1)
    )

    if real_std > 0.0:
        variability_reduction = (
            1.0
            - augmented_std
            / real_std
        ) * 100.0
    else:
        variability_reduction = np.nan

    stability_records.append({
        "metric":
            metric_name,

        "real_only_mean":
            float(
                real_values.mean()
            ),

        "augmented_mean":
            float(
                augmented_values.mean()
            ),

        "real_only_standard_deviation":
            real_std,

        "augmented_standard_deviation":
            augmented_std,

        "variability_reduction_percent":
            variability_reduction,

        "real_only_worst_case":
            float(
                real_values.min()
            ),

        "augmented_worst_case":
            float(
                augmented_values.min()
            ),

        "worst_case_improvement":
            float(
                augmented_values.min()
                - real_values.min()
            )
    })

stability_summary_df = pd.DataFrame(
    stability_records
)

stability_summary_df.to_csv(
    MULTISEED_OUTPUT_DIR
    / "stability_summary.csv",
    index=False
)

display(
    stability_summary_df.style.format({
        "real_only_mean": "{:.4f}",
        "augmented_mean": "{:.4f}",
        "real_only_standard_deviation":
            "{:.4f}",
        "augmented_standard_deviation":
            "{:.4f}",
        "variability_reduction_percent":
            "{:.2f}%",
        "real_only_worst_case":
            "{:.4f}",
        "augmented_worst_case":
            "{:.4f}",
        "worst_case_improvement":
            "{:+.4f}"
    })
)

### Improvement by seed

Positive bars favor synthetic augmentation; negative bars favor the corresponding real-only run.


In [ ]:
selected_improvement_metrics = [
    "balanced_accuracy",
    "macro_f1",
    "bird_f1",
    "roc_auc"
]

paired_improvement_plot_df = (
    paired_seed_metrics_df[
        paired_seed_metrics_df[
            "metric"
        ].isin(
            selected_improvement_metrics
        )
    ]
    .copy()
)

metric_display_names = {
    "balanced_accuracy":
        "Balanced accuracy",

    "macro_f1":
        "Macro-F1",

    "bird_f1":
        "Bird F1",

    "roc_auc":
        "ROC-AUC"
}

paired_improvement_plot_df[
    "metric_display_name"
] = (
    paired_improvement_plot_df[
        "metric"
    ].map(
        metric_display_names
    )
)

plt.figure(figsize=(11, 6))

sns.barplot(
    data=paired_improvement_plot_df,
    x="metric_display_name",
    y="improvement",
    hue="seed",
    palette="tab10"
)

plt.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=1
)

plt.xlabel("Test metric")
plt.ylabel(
    "Augmented − real-only score"
)

plt.title(
    "Paired Synthetic-Augmentation "
    "Improvement across Training Seeds"
)

plt.legend(
    title="Seed",
    ncol=5,
    loc="upper right"
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()
plt.show()

### Saved experiment manifest

The final configuration and primary results are recorded for reproducible downstream analysis.


In [ ]:
multiseed_manifest = {
    "experiment":
        "multi_seed_augmentation_robustness",

    "seeds":
        EXPERIMENT_SEEDS,

    "number_of_seeds":
        len(EXPERIMENT_SEEDS),

    "reference_seed_reused":
        REFERENCE_SEED,

    "newly_trained_seeds":
        NEW_TRAINING_SEEDS,

    "configurations": {
        "real_only": {
            "real_observations":
                1150,

            "synthetic_observations":
                0
        },

        "real_plus_synthetic": {
            "real_observations":
                1150,

            "synthetic_observations":
                1150
        }
    },

    "model_parameters":
        29121,

    "threshold_selection":
        (
            "Maximum validation macro-F1; "
            "balanced accuracy and accuracy "
            "used as tie-breakers"
        ),

    "test_partition_used_for_selection":
        False,

    "primary_metric":
        "macro_f1",

    "mean_macro_f1_real_only":
        float(
            paired_improvement_summary_df
            .set_index("metric")
            .loc[
                "macro_f1",
                "real_only_mean"
            ]
        ),

    "mean_macro_f1_augmented":
        float(
            paired_improvement_summary_df
            .set_index("metric")
            .loc[
                "macro_f1",
                "augmented_mean"
            ]
        ),

    "mean_paired_macro_f1_improvement":
        float(
            paired_improvement_summary_df
            .set_index("metric")
            .loc[
                "macro_f1",
                "mean_paired_improvement"
            ]
        )
}

with open(
    MULTISEED_OUTPUT_DIR
    / "multiseed_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        multiseed_manifest,
        file,
        indent=2
    )

print(
    "Multi-seed experiment manifest saved."
)

## 8. Discussion, Limitations, and Conclusion

### Multi-seed performance

Across five paired training seeds, the 10% real-only configuration achieved a mean test accuracy of **0.8794 ± 0.0705**, balanced accuracy of **0.8518 ± 0.0711**, macro-F1 of **0.8003 ± 0.0999**, bird F1 of **0.6749 ± 0.1552**, and ROC-AUC of **0.9276 ± 0.0496**.

The 1:1 synthetic-augmented configuration achieved a mean test accuracy of **0.9598 ± 0.0296**, balanced accuracy of **0.9268 ± 0.0405**, macro-F1 of **0.9209 ± 0.0536**, bird F1 of **0.8654 ± 0.0895**, and ROC-AUC of **0.9863 ± 0.0182**.

### Paired improvements

Synthetic augmentation produced a mean paired improvement of:

- **+0.0804** in accuracy;
- **+0.0750** in balanced accuracy;
- **+0.1206** in macro-F1;
- **+0.2650** in bird precision;
- **+0.0674** in bird recall;
- **+0.1905** in bird F1;
- **+0.0826** in drone recall;
- **+0.0587** in ROC-AUC.

Macro-F1, accuracy, bird precision, bird F1, drone recall, and ROC-AUC improved for all five paired seeds. Balanced accuracy and bird recall improved for four of five seeds. The single balanced-accuracy and bird-recall reductions occurred for seed 62, where the real-only model already achieved unusually strong performance.

### Uncertainty

The 95% t-based interval for the paired macro-F1 improvement was **[+0.0343, +0.2069]**, and the corresponding interval for balanced accuracy was **[+0.0091, +0.1409]**. The bird-recall interval was **[−0.0195, +0.1543]**, so the direction of the bird-recall effect is less certain than the effects on bird precision or bird F1.

These intervals are descriptive uncertainty estimates rather than definitive population-level claims because only five seed pairs were evaluated and the t-based calculation assumes an approximately normal distribution of paired differences.

### Training stability

The real-only model exhibited substantial sensitivity to initialization and training order. Its macro-F1 ranged from 0.6721 to 0.9491, whereas augmented macro-F1 ranged from 0.8255 to 0.9512. Augmentation therefore improved both average and worst-case performance, although it did not eliminate training instability.

Seed 72 remained the weakest augmented run, demonstrating that synthetic augmentation cannot guarantee successful optimization for every initialization. Nevertheless, it raised macro-F1 from 0.6721 to 0.8255 for that paired seed.

### Scientific interpretation

The multi-seed experiment strengthens the conclusion from the seed-42 analysis. The gain from augmentation is not restricted to one favorable model initialization. Its most consistent effects are higher bird precision, bird F1, macro-F1, drone recall, and ROC-AUC, together with lower performance variability.

The augmented training set contains 2,300 observations but only 1,150 independent real parent samples. Therefore, the result demonstrates the value of transformation-based augmentation, not equivalence to collecting 1,150 additional independent real measurements.

### Limitations

1. Only five training seeds were evaluated.
2. The real subset and synthetic dataset were fixed across seeds.
3. The official split is segment-based rather than session-independent.
4. The test set was reused across models for comparative evaluation.
5. Synthetic children remain correlated with their real parents.
6. Only the 1:1 synthetic-to-real ratio was evaluated.
7. The experiment measures training stochasticity, not variability in real-subset selection or synthetic-data generation.

### Conclusion

Transformation-based synthetic augmentation provides a reproducible improvement in low-data drone–bird micro-Doppler classification. Across five paired seeds, it increased mean macro-F1 from 0.8003 to 0.9209 and mean balanced accuracy from 0.8518 to 0.9268. It also substantially improved minority-class bird F1 and reduced the severity and variability of weak training runs.

The evidence supports retaining synthetic augmentation as a data-efficiency strategy. The next experiments should investigate augmentation-ratio sensitivity and session-independent generalization.